# 🧬 Variant Calling Pipeline — BRCA1 (Google Colab)

This notebook implements a **variant calling pipeline** for Illumina paired-end sequencing data using the BRCA1 gene as a case study.

Two complementary variant calling approaches are provided side-by-side:

| Approach | Caller | Best for |
|----------|--------|----------|
| **A** | GATK HaplotypeCaller | Clinical / germline (GATK Best Practices) |
| **B** | FreeBayes | Research / flexible allele-frequency analysis |

---

## 📚 Pipeline overview

```
Raw FASTQ (paired-end)
    │
    ├── [Step 1] Download data            → Entrez (chr17 GRCh38) + GitHub (BRCA1)
    ├── [Step 2] Quality Control          → FastQC + MultiQC
    ├── [Step 3] Trimming                 → fastp (Q≥20)
    ├── [Step 4] Alignment                → BWA-MEM → sorted, indexed BAM
    ├── [Step 5A] Variant Calling — GATK  → HaplotypeCaller → GVCF → VCF
    ├── [Step 5B] Variant Calling — Free  → FreeBayes → VCF
    ├── [Step 6] Normalization (FreeBayes)→ rename chr + bcftools norm
    ├── [Step 7] Annotation               → bcftools annotate + ClinVar mini
    └── [Step 8] Summary Statistics       → bcftools stats (SNP/INDEL, Ts/Tv)
```

## 🗂️ Input data

- **Samples**: BRCA1_WT, BRCA1_185delAG, BRCA1_c.5266dupC (from [oncogensus/Curso-RSG-Brazil](https://github.com/oncogensus/Curso-RSG-Brazil))
- **Reference**: chromosome 17 GRCh38 (NC_000017.11, downloaded via Entrez)
- **Annotation**: ClinVar mini database (provided in the repository)

> 💡 Run cells sequentially. After **Step 0**, wait for the Colab kernel to restart before continuing.

## ⚙️ Step 0 — Environment setup (condacolab + mamba)

In [1]:
# Install condacolab and initialise mamba (run once per Colab session)
!pip install --upgrade --force-reinstall zstandard --quiet
!pip install -q condacolab

import condacolab
condacolab.install()

# Remove version pins that may cause conflicts
!sed -i '/cudatoolkit/d' /usr/local/conda-meta/pinned
!sed -i '/python/d' /usr/local/conda-meta/pinned

# Install curl (needed by some tools)
!mamba install -c conda-forge curl -y --quiet

!echo "Setup complete"
!mamba env list

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 87.1 MB/s eta 0:00:00
✨🍰✨ Everything looks OK!
Setup complete

# conda environments:
#
base                   /usr/local



## 📥 Step 1 — Download reference genome and sequencing data

In [2]:
# Create environment to download the reference genome via Entrez (NCBI)
!mamba create -n entrez -y --quiet
!mamba install -n entrez -c bioconda entrez-direct -y --quiet

# Create output folders
!mkdir -p refGen data

# Download chromosome 17 (GRCh38)
!echo "# Downloading chromosome 17 (GRCh38) via Entrez"
!mamba run -n entrez bash -c \
  'esearch -db nucleotide -query "NC_000017.11" | efetch -format fasta > ./refGen/chr17_GRCh38.fasta'

!ls -lh ./refGen/

# Clone the course repository with BRCA1 FASTQ files and ClinVar mini database
!echo "# Cloning BRCA1 data from GitHub"
!git clone https://github.com/oncogensus/Curso-RSG-Brazil.git ./brca1
!ls ./brca1/

Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done
# Downloading chromosome 17 (GRCh38) via Entrez
curl: (56) OpenSSL SSL_read: OpenSSL/3.6.2: error:0A000126:SSL routines::unexpected eof while reading, errno 0
 ERROR:  curl command failed ( Tue Jun  9 04:49:57 PM UTC 2026 ) with: 56
-X POST https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi -d retmax=0&usehistory=y&db=nuccore&term=NC_000017.11&tool=edirect&edirect=25.3&edirect_os=Linux
HTTP/1.0 200 OK
curl: (56) OpenSSL SSL_read: OpenSSL/3.6.2: error:0A000126:SSL routines::unexpected eof while reading, errno 0
 ERROR:  curl command failed ( Tue Jun  9 04:49:58 PM UTC 2026 ) with: 56
-X POST https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi -d query_key=1&WebEnv=MCID_6a284435cf42b84fd303c950&retstart=0&retmax=1&db=nuccore&r

## 🔬 Step 2 — Quality control (FastQC + MultiQC)

In [3]:
# Create environment with FastQC, MultiQC and fastp
!mamba create -n quality -c bioconda fastqc multiqc fastp -y --quiet

!mkdir -p fastqc multiqc trimmed

Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done


In [4]:
# Run FastQC on all samples
!echo "# Running FastQC"
!mamba run -n quality fastqc ./brca1/*.fq.gz -o ./fastqc --quiet

# Run MultiQC to aggregate reports
!echo "# Running MultiQC"
!mamba run -n quality multiqc ./fastqc -o ./multiqc --quiet

# Download MultiQC report
from google.colab import files
files.download("./multiqc/multiqc_report.html")

# Running FastQC
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip

# Running MultiQC


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ✂️ Step 3 — Read trimming (fastp, Q≥20)

In [5]:
# Trim all three samples (Q≥20, auto adapter detection)
!echo "# Trimming BRCA1_WT"
!mamba run -n quality fastp \
  -i ./brca1/BRCA1_WT_R1.fq.gz \
  -I ./brca1/BRCA1_WT_R2.fq.gz \
  --detect_adapter_for_pe \
  -o ./trimmed/BRCA1_WT_R1_trim.fq.gz \
  -O ./trimmed/BRCA1_WT_R2_trim.fq.gz \
  -h ./trimmed/BRCA1_WT_fastp.html \
  --qualified_quality_phred 20 --thread 4

!echo "# Trimming BRCA1_185delAG"
!mamba run -n quality fastp \
  -i ./brca1/BRCA1_185delAG_R1.fq.gz \
  -I ./brca1/BRCA1_185delAG_R2.fq.gz \
  --detect_adapter_for_pe \
  -o ./trimmed/BRCA1_185delAG_R1_trim.fq.gz \
  -O ./trimmed/BRCA1_185delAG_R2_trim.fq.gz \
  -h ./trimmed/BRCA1_185delAG_fastp.html \
  --qualified_quality_phred 20 --thread 4

!echo "# Trimming BRCA1_c.5266dupC"
!mamba run -n quality fastp \
  -i ./brca1/BRCA1_c.5266dupC_R1.fq.gz \
  -I ./brca1/BRCA1_c.5266dupC_R2.fq.gz \
  --detect_adapter_for_pe \
  -o ./trimmed/BRCA1_c.5266dupC_R1_trim.fq.gz \
  -O ./trimmed/BRCA1_c.5266dupC_R2_trim.fq.gz \
  -h ./trimmed/BRCA1_c.5266dupC_fastp.html \
  --qualified_quality_phred 20 --thread 4

!ls -lh ./trimmed/*.fq.gz

# Trimming BRCA1_WT
Detecting adapter sequence for read1...
No adapter detected for read1

Detecting adapter sequence for read2...
No adapter detected for read2

Read1 before filtering:
total reads: 12910
total bases: 1936500
Q20 bases: 1898131(98.0186%)
Q30 bases: 1775298(91.6756%)
Q40 bases: 380897(19.6694%)

Read2 before filtering:
total reads: 12910
total bases: 1936500
Q20 bases: 1884320(97.3054%)
Q30 bases: 1735264(89.6083%)
Q40 bases: 379673(19.6061%)

Read1 after filtering:
total reads: 12910
total bases: 1936500
Q20 bases: 1898131(98.0186%)
Q30 bases: 1775298(91.6756%)
Q40 bases: 380897(19.6694%)

Read2 after filtering:
total reads: 12910
total bases: 1936500
Q20 bases: 1884320(97.3054%)
Q30 bases: 1735264(89.6083%)
Q40 bases: 379673(19.6061%)

Filtering result:
reads passed filter: 25820
reads failed due to low quality: 0
reads failed due to too many N: 0
reads failed due to too short: 0
reads failed due to adapter dimer: 0
reads with adapter trimmed: 0
bases trimmed due to a

## 🧭 Step 4 — Alignment (BWA-MEM → sorted, indexed BAM)

**BWA-MEM** aligns reads to the reference genome.  
**Samtools** converts SAM to BAM, sorts by coordinate, and creates an index.

Read groups (`-R`) are required by GATK and help track sample identity.

> ⚠️ After alignment we also extract **only mapped reads** (`-F 4`) to reduce file size and avoid issues in variant calling.

In [7]:
# Create environments for BWA and Samtools
!mamba create -n bwa -c bioconda bwa -y --quiet
!mamba create -n samtools -c bioconda samtools -y --quiet

Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done


In [8]:
# Index the reference genome
!echo "# Indexing reference genome"
!mamba run -n bwa bwa index -p ./refGen/chr17_GRCh38 ./refGen/chr17_GRCh38.fasta

# Create Samtools FASTA index (required by GATK)
!mamba run -n samtools samtools faidx ./refGen/chr17_GRCh38.fasta

# Indexing reference genome
[bwa_index] Pack FASTA... 0.63 sec
[bwa_index] Construct BWT for the packed sequence...
[BWTIncCreate] textLength=166514882, availableWord=23716276
[BWTIncConstructFromPacked] 10 iterations done. 39120754 characters processed.
[BWTIncConstructFromPacked] 20 iterations done. 72271554 characters processed.
[BWTIncConstructFromPacked] 30 iterations done. 101732034 characters processed.
[BWTIncConstructFromPacked] 40 iterations done. 127912546 characters processed.
[BWTIncConstructFromPacked] 50 iterations done. 151177842 characters processed.
[bwt_gen] Finished constructing BWT in 58 iterations.
[bwa_index] 45.22 seconds elapse.
[bwa_index] Update BWT... 0.43 sec
[bwa_index] Pack forward-only FASTA... 0.40 sec
[bwa_index] Construct SA from BWT and Occ... 24.11 sec
[main] Version: 0.7.19-r1273
[main] CMD: bwa index -p ./refGen/chr17_GRCh38 ./refGen/chr17_GRCh38.fasta
[main] Real time: 72.029 sec; CPU: 70.798 sec



In [9]:
%%bash
# Align each sample: BWA-MEM → sort → index → flagstat
mkdir -p aligned

for SAMPLE in BRCA1_WT BRCA1_185delAG BRCA1_c.5266dupC; do
  R1="./trimmed/${SAMPLE}_R1_trim.fq.gz"
  R2="./trimmed/${SAMPLE}_R2_trim.fq.gz"
  RAW_BAM="./aligned/${SAMPLE}.bam"
  SORTED_BAM="./aligned/${SAMPLE}_sorted.bam"
  MAPPED_BAM="./aligned/${SAMPLE}_mapped.bam"

  echo "--- Aligning ${SAMPLE} ---"
  mamba run -n bwa bwa mem -M -t 4 \
    -R "@RG\tID:${SAMPLE}\tSM:${SAMPLE}\tPL:ILLUMINA\tLB:${SAMPLE}_lib1\tPU:${SAMPLE}_unit1" \
    ./refGen/chr17_GRCh38 $R1 $R2 -o $RAW_BAM

  echo "--- Sorting ${SAMPLE} ---"
  mamba run -n samtools samtools sort -o $SORTED_BAM $RAW_BAM

  echo "--- Indexing ${SAMPLE} ---"
  mamba run -n samtools samtools index $SORTED_BAM

  echo "--- Flagstat ${SAMPLE} ---"
  mamba run -n samtools samtools flagstat $SORTED_BAM

  echo "--- Extracting mapped reads only (removing flag 4) ---"
  mamba run -n samtools samtools view -b -F 4 -o $MAPPED_BAM $SORTED_BAM
  mamba run -n samtools samtools index $MAPPED_BAM
done

echo "Aligned BAM files:"
ls -lh ./aligned/*_mapped.bam

--- Aligning BRCA1_WT ---
--- Sorting BRCA1_WT ---
--- Indexing BRCA1_WT ---
--- Flagstat BRCA1_WT ---
25820 + 0 in total (QC-passed reads + QC-failed reads)
25820 + 0 primary
0 + 0 secondary
0 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
25820 + 0 mapped (100.00% : N/A)
25820 + 0 primary mapped (100.00% : N/A)
25820 + 0 paired in sequencing
12910 + 0 read1
12910 + 0 read2
25818 + 0 properly paired (99.99% : N/A)
25820 + 0 with itself and mate mapped
0 + 0 singletons (0.00% : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)

--- Extracting mapped reads only (removing flag 4) ---
--- Aligning BRCA1_185delAG ---
--- Sorting BRCA1_185delAG ---
--- Indexing BRCA1_185delAG ---
--- Flagstat BRCA1_185delAG ---
25820 + 0 in total (QC-passed reads + QC-failed reads)
25820 + 0 primary
0 + 0 secondary
0 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
25820 + 0 mapped (100.00% : N/A)
25820 + 0 primary mapped (100.00% : N/

[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::process] read 25820 sequences (3873000 bp)...
[M::mem_pestat] # candidate unique pairs for (FF, FR, RF, RR): (0, 12460, 0, 0)
[M::mem_pestat] skip orientation FF as there are not enough pairs
[M::mem_pestat] analyzing insert size distribution for orientation FR...
[M::mem_pestat] (25, 50, 75) percentile: (192, 198, 205)
[M::mem_pestat] low and high boundaries for computing mean and std.dev: (166, 231)
[M::mem_pestat] mean and std.dev: (198.55, 9.89)
[M::mem_pestat] low and high boundaries for proper pairs: (153, 244)
[M::mem_pestat] skip orientation RF as there are not enough pairs
[M::mem_pestat] skip orientation RR as there are not enough pairs
[M::mem_process_seqs] Processed 25820 reads in 6.627 CPU sec, 3.508 real sec
[main] Version: 0.7.19-r1273
[main] CMD: bwa mem -M -t 4 -R @RG\tID:BRCA1_WT\tSM:BRCA1_WT\tPL:ILLUMINA\tLB:BRCA1_WT_lib1\tPU:BRCA1_WT_unit1 -o ./aligned/BRCA1_WT.bam ./refGen/chr17_GRCh38 ./trimmed/BRCA1_WT_R1_trim.fq

## 🧪 Step 5A — Variant Calling: GATK HaplotypeCaller

**Workflow:** HaplotypeCaller (GVCF) → CombineGVCFs → GenotypeGVCFs → VariantFiltration

In [11]:
# Install GATK4
!mamba create -n gatk -c bioconda gatk4 -y --quiet

Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done


In [13]:
# Create sequence dictionary (required by GATK)
!mamba run -n gatk gatk CreateSequenceDictionary \
  -R /content/refGen/chr17_GRCh38.fasta \
  -O /content/refGen/chr17_GRCh38.dict

!mkdir -p variants/gatk

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

Using GATK jar /usr/local/envs/gatk/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /usr/local/envs/gatk/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar CreateSequenceDictionary -R /content/refGen/chr17_GRCh38.fasta -O /content/refGen/chr17_GRCh38.dict
17:02:44.585 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/usr/local/envs/gatk/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so

[Tue Jun 09 17:02:44 UTC 

In [14]:
%%bash
# HaplotypeCaller in GVCF mode for each sample
for SAMPLE in BRCA1_WT BRCA1_185delAG BRCA1_c.5266dupC; do
  echo "--- HaplotypeCaller: ${SAMPLE} ---"
  mamba run -n gatk gatk HaplotypeCaller \
    -R /content/refGen/chr17_GRCh38.fasta \
    -I ./aligned/${SAMPLE}_mapped.bam \
    -O ./variants/gatk/${SAMPLE}.g.vcf.gz \
    -ERC GVCF --verbosity ERROR
done

--- HaplotypeCaller: BRCA1_WT ---
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

--- HaplotypeCaller: BRCA1_185delAG ---
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

--- HaplotypeCaller: BRCA1_c.5266dupC ---
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path

Using GATK jar /usr/local/envs/gatk/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /usr/local/envs/gatk/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar HaplotypeCaller -R /content/refGen/chr17_GRCh38.fasta -I ./aligned/BRCA1_WT_mapped.bam -O ./variants/gatk/BRCA1_WT.g.vcf.gz -ERC GVCF --verbosity ERROR

[June 9, 2026, 5:04:45 PM UTC] org.broadinstitute.hellbender.tools.walkers.haplotypecaller.HaplotypeCaller done. Elapsed time: 1.28 minutes.
Runtime.totalMemory()=1457520640

Using GATK jar /usr/local/envs/gatk/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /usr/local/envs/gatk/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-

In [15]:
# Combine GVCFs → joint genotyping → filter
!echo "# CombineGVCFs"
!mamba run -n gatk gatk CombineGVCFs \
  -R /content/refGen/chr17_GRCh38.fasta \
  -V ./variants/gatk/BRCA1_WT.g.vcf.gz \
  -V ./variants/gatk/BRCA1_185delAG.g.vcf.gz \
  -V ./variants/gatk/BRCA1_c.5266dupC.g.vcf.gz \
  -O ./variants/gatk/cohort.g.vcf.gz --verbosity ERROR

!echo "# GenotypeGVCFs"
!mamba run -n gatk gatk GenotypeGVCFs \
  -R /content/refGen/chr17_GRCh38.fasta \
  -V ./variants/gatk/cohort.g.vcf.gz \
  -O ./variants/gatk/cohort.vcf.gz --verbosity ERROR

!echo "# VariantFiltration (QD < 2, FS > 60, MQ < 40)"
!mamba run -n gatk gatk VariantFiltration \
  -R /content/refGen/chr17_GRCh38.fasta \
  -V ./variants/gatk/cohort.vcf.gz \
  -O ./variants/gatk/cohort_filtered.vcf.gz \
  --filter-expression "QD < 2.0 || FS > 60.0 || MQ < 40.0" \
  --filter-name "GATK_filter" --verbosity ERROR

!echo "# GATK VCF preview:"
!zcat ./variants/gatk/cohort_filtered.vcf.gz | grep -v "^##" | head -20

# CombineGVCFs
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

Using GATK jar /usr/local/envs/gatk/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /usr/local/envs/gatk/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar CombineGVCFs -R /content/refGen/chr17_GRCh38.fasta -V ./variants/gatk/BRCA1_WT.g.vcf.gz -V ./variants/gatk/BRCA1_185delAG.g.vcf.gz -V ./variants/gatk/BRCA1_c.5266dupC.g.vcf.gz -O ./variants/gatk/cohort.g.vcf.gz --verbosity ERROR

[June 9, 2026, 5:07:40 PM UTC] org.broadinstitute.hellbender.tools.walkers.CombineGV

## 🧪 Step 5B — Variant Calling: FreeBayes

**Workflow:** FreeBayes (multi-BAM) → bcftools filter (QUAL≥20)

In [16]:
# Install FreeBayes and bcftools
!mamba create -n freebayes -c bioconda freebayes bcftools -y --quiet

!mkdir -p variants/freebayes

Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done


In [18]:
%%bash
# FreeBayes: joint multi-sample calling from all mapped BAMs at once
mamba run -n freebayes freebayes \
  -f ./refGen/chr17_GRCh38.fasta \
  ./aligned/BRCA1_WT_mapped.bam \
  ./aligned/BRCA1_185delAG_mapped.bam \
  ./aligned/BRCA1_c.5266dupC_mapped.bam > ./variants/freebayes/cohort_freebayes.vcf

mamba run -n freebayes bcftools index -t ./variants/freebayes/cohort_freebayes.vcf

echo "# Filtering: QUAL >= 20, SNPs and INDELs only"
mamba run -n freebayes bash -c \
  'bcftools view -v snps,indels ./variants/freebayes/cohort_freebayes.vcf | \
   bcftools filter -i "QUAL>=20" -Oz -o ./variants/freebayes/cohort_freebayes_filtered.vcf.gz'

mamba run -n freebayes bcftools index -t ./variants/freebayes/cohort_freebayes_filtered.vcf.gz

echo "# FreeBayes VCF preview:"
zcat ./variants/freebayes/cohort_freebayes_filtered.vcf.gz | grep -v "^##" | head -10

# Filtering: QUAL >= 20, SNPs and INDELs only
# FreeBayes VCF preview:
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	BRCA1_c.5266dupC	BRCA1_185delAG	BRCA1_WT
NC_000017.11	43155468	.	C	A	1523	PASS	AB=0;ABP=0;AC=6;AF=1;AN=6;AO=47;CIGAR=1X;DP=47;DPB=47;DPRA=0;EPP=4.16534;EPPR=0;GTI=0;LEN=1;MEANALT=1;MQM=60;MQMR=0;NS=3;NUMALT=1;ODDS=23.1015;PAIRED=1;PAIREDR=0;PAO=0;PQA=0;PQR=0;PRO=0;QA=1718;QR=0;RO=0;RPL=28;RPP=6.75262;RPPR=0;RPR=19;RUN=1;SAF=24;SAP=3.0565;SAR=23;SRF=0;SRP=0;SRR=0;TYPE=snp;technology.ILLUMINA=1	GT:DP:AD:RO:QR:AO:QA:GL	1/1:15:0,15:0:0:15:553:-50.0978,-4.51545,0	1/1:20:0,20:0:0:20:713:-64.4792,-6.0206,0	1/1:12:0,12:0:0:12:452:-41.0185,-3.61236,0
NC_000017.11	43201094	.	C	G	2454.09	PASS	AB=0;ABP=0;AC=6;AF=1;AN=6;AO=76;CIGAR=1X;DP=76;DPB=76;DPRA=0;EPP=3.46745;EPPR=0;GTI=0;LEN=1;MEANALT=1;MQM=60;MQMR=0;NS=3;NUMALT=1;ODDS=30.4683;PAIRED=1;PAIREDR=0;PAO=0;PQA=0;PQR=0;PRO=0;QA=2754;QR=0;RO=0;RPL=41;RPP=4.03889;RPPR=0;RPR=35;RUN=1;SAF=35;SAP=4.03889;SAR=41;SRF=0;SRP=0;SRR=0;TYPE=sn

index: the file is not BGZF compressed, cannot index: ./variants/freebayes/cohort_freebayes.vcf

ERROR conda.cli.main_run:execute(125): `conda run bcftools index -t ./variants/freebayes/cohort_freebayes.vcf` failed. (See above for error)
[W::bcf_hdr_check_sanity] GQ should be declared as Type=Integer
[W::vcf_parse] Contig '' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::bcf_hrec_check] Invalid contig name: ""
Error: VCF parse error
[W::bcf_hdr_check_sanity] GQ should be declared as Type=Integer



## 🔧 Step 6 — Normalise FreeBayes VCF for annotation

> **Why is this step needed?**
>
> The `bcftools annotate` command matches variants by **exact position + chromosome name + allele**.  
> Two differences between the FreeBayes output and the ClinVar mini database break this matching:
>
> | Issue | FreeBayes output | ClinVar mini expects |
> |-------|-----------------|----------------------|
> | Chromosome name | `17` (from Entrez FASTA header) | `chr17` |
> | INDEL representation | May differ (right-aligned) | Left-aligned (bcftools norm) |
>
> This step fixes both issues before annotation, so FreeBayes variants receive the same ClinVar IDs as GATK.

In [19]:
%%bash
# Create a chromosome name mapping file: 17 → chr17
echo "17	chr17" > /tmp/chr_rename.txt

# Rename chromosome names (17 → chr17)
mamba run -n freebayes bcftools annotate \
  --rename-chrs /tmp/chr_rename.txt \
  ./variants/freebayes/cohort_freebayes_filtered.vcf.gz \
  -Oz -o ./variants/freebayes/cohort_freebayes_chr.vcf.gz

mamba run -n freebayes bcftools index -t ./variants/freebayes/cohort_freebayes_chr.vcf.gz

# Normalise INDEL representation (left-align + split multiallelic)
mamba run -n freebayes bcftools norm \
  -m-any \
  -f ./refGen/chr17_GRCh38.fasta \
  ./variants/freebayes/cohort_freebayes_chr.vcf.gz \
  -Oz -o ./variants/freebayes/cohort_freebayes_norm.vcf.gz

mamba run -n freebayes bcftools index -t ./variants/freebayes/cohort_freebayes_norm.vcf.gz

echo "# Normalised FreeBayes VCF preview:"
zcat ./variants/freebayes/cohort_freebayes_norm.vcf.gz | grep -v "^##" | head -10

# Normalised FreeBayes VCF preview:
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	BRCA1_c.5266dupC	BRCA1_185delAG	BRCA1_WT
NC_000017.11	43155468	.	C	A	1523	PASS	AB=0;ABP=0;AC=6;AF=1;AN=6;AO=47;CIGAR=1X;DP=47;DPB=47;DPRA=0;EPP=4.16534;EPPR=0;GTI=0;LEN=1;MEANALT=1;MQM=60;MQMR=0;NS=3;NUMALT=1;ODDS=23.1015;PAIRED=1;PAIREDR=0;PAO=0;PQA=0;PQR=0;PRO=0;QA=1718;QR=0;RO=0;RPL=28;RPP=6.75262;RPPR=0;RPR=19;RUN=1;SAF=24;SAP=3.0565;SAR=23;SRF=0;SRP=0;SRR=0;TYPE=snp;technology.ILLUMINA=1	GT:DP:AD:RO:QR:AO:QA:GL	1/1:15:0,15:0:0:15:553:-50.0978,-4.51545,0	1/1:20:0,20:0:0:20:713:-64.4792,-6.0206,0	1/1:12:0,12:0:0:12:452:-41.0185,-3.61236,0
NC_000017.11	43201094	.	C	G	2454.09	PASS	AB=0;ABP=0;AC=6;AF=1;AN=6;AO=76;CIGAR=1X;DP=76;DPB=76;DPRA=0;EPP=3.46745;EPPR=0;GTI=0;LEN=1;MEANALT=1;MQM=60;MQMR=0;NS=3;NUMALT=1;ODDS=30.4683;PAIRED=1;PAIREDR=0;PAO=0;PQA=0;PQR=0;PRO=0;QA=2754;QR=0;RO=0;RPL=41;RPP=4.03889;RPPR=0;RPR=35;RUN=1;SAF=35;SAP=4.03889;SAR=41;SRF=0;SRP=0;SRR=0;TYPE=snp;technology.ILLUMINA=1	GT:DP:AD:RO

[W::bcf_hdr_check_sanity] GQ should be declared as Type=Integer

[W::bcf_hdr_check_sanity] GQ should be declared as Type=Integer
Lines   total/split/joined/realigned/mismatch_removed/dup_removed/skipped:	12/0/0/6/0/0/0



## 🧬 Step 7 — Variant annotation (bcftools + ClinVar mini)

`bcftools annotate` matches variants by chromosome + position + allele and adds:
- **ID**: ClinVar variant ID or rsID
- **INFO**: clinical significance, gene name, condition

In [20]:
# Install bcftools in its own environment
!mamba create -n bcftools -c bioconda bcftools -y --quiet

!mkdir -p annotated

Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done


In [25]:
# --- Annotate GATK output ---
!echo "# Annotating GATK VCF"
!mamba run -n bcftools bcftools annotate \
  -a ./brca1/clinvar_mini.vcf.gz \
  -c ID,INFO \
  ./variants/gatk/cohort_filtered.vcf.gz \
  -o ./annotated/cohort_gatk_annotated.vcf

# --- Annotate normalised FreeBayes output ---
!echo "# Annotating FreeBayes VCF (normalised)"
!mamba run -n bcftools bcftools annotate \
  -a ./brca1/clinvar_mini.vcf.gz \
  -c ID,INFO \
  ./variants/freebayes/cohort_freebayes_norm.vcf.gz \
  -o ./annotated/cohort_freebayes_annotated.vcf

# Annotating GATK VCF
# Annotating FreeBayes VCF (normalised)
[W::bcf_hdr_check_sanity] GQ should be declared as Type=Integer



In [24]:
!echo "# Annotated GATK VCF:"
!grep -v "^##" ./annotated/cohort_gatk_annotated.vcf | head -20

!echo "# Annotated FreeBayes VCF:"
!grep -v "^##" ./annotated/cohort_freebayes_annotated.vcf | head -20

# Annotated GATK VCF:
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	BRCA1_185delAG	BRCA1_WT	BRCA1_c.5266dupC
NC_000017.11	43155468	.	C	A	1457.73	PASS	AC=6;AF=1;AN=6;DP=51;ExcessHet=0;FS=0;MLEAC=6;MLEAF=1;MQ=60;QD=31.02;SOR=0.735	GT:AD:DP:GQ:PL	1/1:0,20:20:60:593,60,0	1/1:0,12:12:36:378,36,0	1/1:0,15:15:45:500,45,0
NC_000017.11	43201094	.	C	G	2245.73	PASS	AC=6;AF=1;AN=6;DP=76;ExcessHet=0;FS=0;MLEAC=6;MLEAF=1;MQ=60;QD=29.55;SOR=0.859	GT:AD:DP:GQ:PL	1/1:0,41:41:99:1177,123,0	1/1:0,20:20:60:548,60,0	1/1:0,15:15:45:534,45,0
NC_000017.11	43201451	.	A	G	2465.73	PASS	AC=6;AF=1;AN=6;DP=85;ExcessHet=0;FS=0;MLEAC=6;MLEAF=1;MQ=60;QD=29.01;SOR=0.867	GT:AD:DP:GQ:PL	1/1:0,32:32:96:966,96,0	1/1:0,25:25:75:712,75,0	1/1:0,28:28:84:801,84,0
NC_000017.11	43202179	.	G	GA	1156.69	PASS	AC=6;AF=1;AN=6;DP=63;ExcessHet=0;FS=0;MLEAC=6;MLEAF=1;MQ=60;QD=20.66;SOR=0.841	GT:AD:DP:GQ:PL	1/1:0,22:22:65:462,65,0	1/1:0,18:18:53:373,53,0	1/1:0,16:16:48:335,48,0
NC_000017.11	43202523	.	TA	T	1363.69	PASS	AC=6;AF=1;AN=6;DP=

## 📊 Step 8 — Summary statistics (bcftools stats)

`bcftools stats` computes:
- Total SNPs and INDELs
- Transition/Transversion ratio (Ts/Tv) — expected ~2.8–3.2 for WES
- Per-sample depth and genotype counts

In [28]:
!echo "=== GATK VCF Statistics ==="
!mamba run -n bcftools bcftools stats \
  ./variants/gatk/cohort_filtered.vcf.gz | grep -E "^(SN|TSTV)" | head -20

!echo "=== FreeBayes VCF Statistics ==="
!mamba run -n bcftools bcftools stats \
  ./variants/freebayes/cohort_freebayes_norm.vcf.gz | grep -E "^(SN|TSTV)" | head -20

=== GATK VCF Statistics ===
SN	0	number of samples:	3
SN	0	number of records:	12
SN	0	number of no-ALTs:	0
SN	0	number of SNPs:	7
SN	0	number of MNPs:	0
SN	0	number of indels:	5
SN	0	number of others:	0
SN	0	number of multiallelic sites:	0
SN	0	number of multiallelic SNP sites:	0
TSTV	0	5	2	2.50	5	2	2.50
=== FreeBayes VCF Statistics ===
[W::bcf_hdr_check_sanity] GQ should be declared as Type=Integer

SN	0	number of samples:	3
SN	0	number of records:	12
SN	0	number of no-ALTs:	0
SN	0	number of SNPs:	7
SN	0	number of MNPs:	0
SN	0	number of indels:	5
SN	0	number of others:	0
SN	0	number of multiallelic sites:	0
SN	0	number of multiallelic SNP sites:	0
TSTV	0	5	2	2.50	5	2	2.50


In [29]:
# Side-by-side comparison using Python
import subprocess

def get_stats(vcf_path):
    result = subprocess.run(
        f"mamba run -n bcftools bcftools stats {vcf_path}",
        shell=True, capture_output=True, text=True
    )
    stats = {}
    for line in result.stdout.split("\n"):
        if line.startswith("SN"):
            parts = line.split("\t")
            if len(parts) >= 4:
                stats[parts[2].strip().rstrip(":")] = parts[3].strip()
    return stats

gatk_stats = get_stats("./variants/gatk/cohort_filtered.vcf.gz")
fb_stats   = get_stats("./variants/freebayes/cohort_freebayes_norm.vcf.gz")

keys = ["number of samples", "number of records", "number of SNPs",
        "number of indels", "number of multiallelic sites", "number of MNPs"]

print(f"{'Metric':<35} {'GATK':>12} {'FreeBayes':>12}")
print("-" * 60)
for k in keys:
    print(f"{k:<35} {gatk_stats.get(k,'N/A'):>12} {fb_stats.get(k,'N/A'):>12}")

Metric                                      GATK    FreeBayes
------------------------------------------------------------
number of samples                              3            3
number of records                             12           12
number of SNPs                                 7            7
number of indels                               5            5
number of multiallelic sites                   0            0
number of MNPs                                 0            0


## 🎉 Pipeline complete!

| Step | Tool | Output |
|------|------|--------|
| Quality Control | FastQC + MultiQC | HTML reports |
| Trimming | fastp | Cleaned FASTQ |
| Alignment | BWA-MEM + Samtools | Sorted, indexed BAM |
| Variant Calling A | GATK HaplotypeCaller | `cohort_filtered.vcf.gz` |
| Variant Calling B | FreeBayes + bcftools | `cohort_freebayes_filtered.vcf.gz` |
| Normalisation | bcftools rename + norm | `cohort_freebayes_norm.vcf.gz` |
| Annotation | bcftools + ClinVar mini | Annotated VCF (both callers) |
| Statistics | bcftools stats | SNP/INDEL counts, Ts/Tv |

```
├── fastqc/               # Per-sample FastQC reports
├── multiqc/              # Aggregated MultiQC report
├── trimmed/              # Trimmed FASTQ files
├── aligned/              # Sorted, indexed, mapped BAM files
├── variants/
│   ├── gatk/             # GATK results
│   └── freebayes/        # FreeBayes results + normalised VCF
└── annotated/            # VCF files annotated with ClinVar (both callers)
```

### 🔗 References
- [GATK Best Practices](https://gatk.broadinstitute.org/hc/en-us/sections/360007226651)
- [FreeBayes GitHub](https://github.com/freebayes/freebayes)
- [bcftools documentation](https://www.htslib.org/doc/bcftools.html)
- [ClinVar](https://www.ncbi.nlm.nih.gov/clinvar/)
- [Original RSG Brazil 2026 notebook](https://github.com/oncogensus/Curso-RSG-Brazil)